# Method A on mjwarp (GPU) — 계산 흐름 추적

Pure Method A coupling이 패치된 mjwarp에서 어떻게 흘러가는지 단일 모터 모델로 직접 확인.

**확인할 것:**
1. β = 1/(1+h/τ) 적분기가 실제로 적용되는지
2. Schur 항 `−(1−β)·Kt·gr·Ke·gr/R`이 implicit solver에 들어가는지
3. RHS 보정 `gain·(1−β)·(ctrl−I_old)`가 force에 더해지는지
4. Demag (Ke_plant ≠ Ke_nom) 시 추가 보정이 작동하는지

In [ ]:
# Source: 자체 작성 (이 노트북 전용 setup).
# 사용 라이브러리:
#   - mujoco                : CPU 모델 빌드 (MjModel.from_xml_string, MjData)
#   - mujoco_warp (mjwarp)  : GPU step 실행 (put_model, put_data, step)
# 패치된 _src 위치는 MJWARP_DIR로 노출 (Cell 6의 grep에 사용).
# 패치 상세 설명: docs/method_a_mjwarp_patch.md
import warp as wp
wp.init()
import mujoco
import mujoco_warp as mjwarp
import numpy as np
import matplotlib.pyplot as plt

MJWARP_DIR = '/home/rbdo/miniconda3/envs/mjlab/lib/python3.11/site-packages/mujoco_warp/_src'
print(f'mjwarp _src: {MJWARP_DIR}')

## 1. Method A 수식 요약

**전기 모터 ODE** (단일 actuator 기준):
$$
L\,\dot I = V - R\,I - K_e\,g_r\,\omega
$$

**제어 변환** (사용자 Python에서 처리): `ctrl = (V − Ke·gr·ω)/R = I_ss`. 즉, ctrl은 "steady-state current"의 의미.

**Method A 적분기** (β = 1/(1+h/τ)):
$$
I_{n+1} = \beta\,I_n + (1-\beta)\,\mathrm{ctrl}, \quad \beta = \frac{1}{1+h/\tau}
$$

**Schur cross-Jacobian** (implicit solver의 좌변에 추가):
$$
\mathrm{qDeriv} \mathrel{+}= -\,(1-\beta)\,\frac{K_t g_r\,K_e g_r}{R}\,J^\top J
$$

→ `M_eff = qM − dt·qDeriv` 가 양정정 강화되어 implicit damping이 됨.

**Force RHS 보정**:
$$
F = K_t g_r\,I_n + K_t g_r\,(1-\beta)\,(\mathrm{ctrl} - I_n) = K_t g_r\,I_{\text{predicted}}
$$

## 2. 최소 모터 모델 빌드

Go2 calf 모터 파라미터: Kt=Ke=0.128, R=0.3, L=1e-4, gear_ratio=6.33.
→ Kt·gr = Ke·gr = 0.81, τ = L/R = 3.33e-4 s.

dt = 0.0001 s (0.1ms) 기준: h/τ = 0.3 → β = 1/1.3 ≈ 0.769, (1−β) ≈ 0.231.

In [ ]:
# Source: 자체 작성. 모터 파라미터 출처:
#   - Kt, Ke, R, L, gear_ratio
#       src/assets/robots/unitree_go2/go2_constants.py:308-330
#         (GO2_COUPLED_ELECTRIC_HIP/THIGH/CALF cfg)
#   - β = 1/(1+h/τ) 정의
#       docs/method_a_mjwarp_patch.md §1
#       (대응 코드: mjwarp/_src/forward.py:147-160)
#   - dynprm 슬롯 의미
#       docs/method_a_mjwarp_patch.md §0
#       (대응 코드: src/assets/robots/unitree_go2/mj_native_electric_actuator.py:272-287)
#   - XML <general> dyntype/dynprm/gainprm/biasprm 사용
#       MuJoCo XML reference (https://mujoco.readthedocs.io/en/stable/XMLreference.html#actuator-general)
Kt = Ke = 0.128
R = 0.3
L = 1e-4
gr = 6.33
tau_e = L / R
dt = 0.0001
Kt_gr = Kt * gr
Ke_gr = Ke * gr
beta_methodA = 1.0 / (1.0 + dt/tau_e)
one_minus_beta = 1.0 - beta_methodA
schur_scale = -(1 - beta_methodA) * Kt_gr * Ke_gr / R

print(f'tau_e         = {tau_e*1e3:.4f} ms')
print(f'h/tau         = {dt/tau_e:.4f}')
print(f'beta (MethodA) = {beta_methodA:.4f}')
print(f'1 - beta       = {one_minus_beta:.4f}')
print(f'Schur scale   = {schur_scale:.6f}  (음수: implicit damping)')

# dynprm 슬롯 의미 (현재 설계):
#   [0] = tau_e
#   [1] = Ke_plant·gr  (demag 시 변경됨; Schur가 이걸 사용)
#   [2] = L
#   [3] = Ke_nom·gr   (demag RHS 보정용; 절대 변경 안 됨)
DYNPRM = f'{tau_e} {Ke_gr} {L} {Ke_gr}'  # healthy: dynprm[1] = dynprm[3]

XML = f'''
<mujoco>
  <option timestep="{dt}" integrator="implicitfast"/>
  <worldbody>
    <body name="link">
      <joint name="j" type="hinge" axis="0 0 1"/>
      <geom type="sphere" size="0.1" mass="1"/>
    </body>
  </worldbody>
  <actuator>
    <general joint="j" dyntype="filterexact"
             dynprm="{DYNPRM}"
             gainprm="{Kt_gr}" biasprm="0 -20 -1"/>
  </actuator>
</mujoco>
'''
m_cpu = mujoco.MjModel.from_xml_string(XML)
print(f'\nactuator_dynprm = {m_cpu.actuator_dynprm[0]}')
print(f'actuator_gainprm = {m_cpu.actuator_gainprm[0]}')
print(f'actuator_biasprm = {m_cpu.actuator_biasprm[0]}')

## 3. 패치된 커널 위치 확인

GPU 한 step에서 다음 순서로 커널이 실행됩니다 (mujoco_warp/_src):

| 순서 | 커널 | 패치 부분 |
|------|------|------|
| 1 | `forward.py:_actuator_force` (act_dot 계산) | demag 보정 (`(dynprm[3]−dynprm[1])·ω/L`)도 여기. 우리가 추가한 RHS 보정도 같은 커널의 force 계산 부분 |
| 2 | `derivative.py:_qderiv_actuator_passive_vel` | Schur scale을 vel에 더함 |
| 3 | `derivative.py:_qderiv_actuator_passive_actuation_sparse` | JᵀBJ 누적 |
| 4 | `derivative.py:_qderiv_actuator_passive` | `M_eff = qM − dt·qDeriv` 계산 |
| 5 | implicit solver | Cholesky → q̈ |
| 6 | `forward.py:_next_act` | β=1/(1+h/τ)로 act 적분 |

In [ ]:
# Source: 자체 작성. grep 대상은 다음 파일들 (이번 패치로 추가된 부분):
#   - mjwarp/_src/forward.py:147-160        ('Pure Method A integrator' marker)
#       → _next_act 함수의 Method A 적분기 분기
#   - mjwarp/_src/forward.py:752-770        ('Method A RHS correction' marker)
#       → _actuator_force 커널의 RHS 보정 추가 부분
#   - mjwarp/_src/derivative.py:67-83       ('Method A Schur' marker)
#       → _qderiv_actuator_passive_vel의 Schur scale 계산 추가
# 패치 전체 구조: docs/method_a_mjwarp_patch.md §1, §2, §3
import subprocess

def show_patch(path, pattern, before=2, after=10):
    out = subprocess.run(
        ['grep', '-n', '-B', str(before), '-A', str(after), pattern, path],
        capture_output=True, text=True
    )
    print(out.stdout)

print('=== forward.py: _next_act (Method A integrator) ===')
show_patch(f'{MJWARP_DIR}/forward.py', 'Pure Method A integrator', before=1, after=8)

print('=== forward.py: _actuator_force (RHS correction) ===')
show_patch(f'{MJWARP_DIR}/forward.py', 'Method A RHS correction', before=0, after=12)

print('=== derivative.py: Schur in vel ===')
show_patch(f'{MJWARP_DIR}/derivative.py', 'Method A Schur', before=0, after=12)

## 4. 단일 step 수동 검증

초기 상태: I=0, ω=0, ctrl=5A.
한 step 후 이론값:
- I_1 = β·0 + (1−β)·5 = 0.231·5 = **1.155 A**
- F = Kt·gr · I_1 = 0.81 · 1.155 ≈ **0.935 N·m** (RHS 보정으로 I_predicted를 사용)
- q̈ ≈ F / I_inertia (sphere I = 2/5·m·r² = 0.004) → ω_1 ≈ q̈·dt

In [ ]:
# Source: 자체 작성. 검증할 식 출처:
#   - I_1 = β·I_0 + (1-β)·ctrl
#       docs/method_a_mjwarp_patch.md §1 (수식 derivation)
#       (대응 코드: mjwarp/_src/forward.py:147-160 _next_act)
#   - F = Kt·gr · I_predicted = Kt·gr · [β·I_0 + (1-β)·ctrl]
#       docs/method_a_mjwarp_patch.md §3 (RHS correction derivation)
#       (대응 코드: mjwarp/_src/forward.py:752-770 _actuator_force)
#   - Sphere inertia I = 2/5·m·r² (관성모멘트 표준식)
m = mujoco.MjModel.from_xml_string(XML)
d = mujoco.MjData(m)
m_warp = mjwarp.put_model(m)
d_warp = mjwarp.put_data(m, d)

# 초기 상태 (I=0, ω=0)
d_warp.ctrl.fill_(5.0)

print('=== 한 step 전 ===')
print(f'  I (act):   {d_warp.act.numpy()[0,0]:.6f}')
print(f'  qvel:      {d_warp.qvel.numpy()[0,0]:.6f}')
print(f'  ctrl:      {d_warp.ctrl.numpy()[0,0]:.6f}')

mjwarp.step(m_warp, d_warp)

print('\n=== 한 step 후 ===')
I_1 = d_warp.act.numpy()[0,0]
qv_1 = d_warp.qvel.numpy()[0,0]
F_1 = d_warp.actuator_force.numpy()[0,0]
print(f'  I (act):       {I_1:.6f}     (이론: {one_minus_beta * 5:.6f})')
print(f'  force:         {F_1:.6f}     (이론: Kt·gr·I_pred = {Kt_gr * one_minus_beta * 5:.6f})')
print(f'  qvel:          {qv_1:.6f}')
print(f'  qacc estimate: {qv_1/dt:.4f} rad/s²')

I_inertia = 2/5 * 1.0 * 0.1**2
print(f'\n  Sphere I_inertia = {I_inertia} kg·m²')
print(f'  expected qacc (no Schur) = F/I = {F_1/I_inertia:.4f} rad/s²')
print(f'  실제 qacc < expected: Schur implicit damping 효과 확인 ✓')

## 5. 다단계 시뮬레이션 — 전류 추적 검증

ctrl=5A 단계 입력 시 I(t)가 Method A 이산식대로 진행하는지 확인.

In [ ]:
# Source: 자체 작성. 비교 대상 이산식 2종:
#   1) Method A:        I_{n+1} = β·I_n + (1-β)·ctrl,   β = 1/(1+h/τ)
#        docs/method_a_mjwarp_patch.md §1
#        (대응 코드: mjwarp/_src/forward.py:147-160)
#   2) Vanilla filterexact: I_{n+1} = β·I_n + (1-β)·ctrl,  β = exp(-h/τ)
#        mjwarp/_src/forward.py.original_vanilla:147-149 (백업 파일)
# 주의: 여기 단순 1차 필터로 모델하는 이유 — 시뮬에서 ctrl을 5A 고정으로 줬고
#      back-EMF coupling을 외부에서 끄지 않았으므로, ω가 작은 초기에는
#      ctrl(ω-의존) 대신 ctrl=상수로 풀어도 거의 동일 (ω 영향은 Schur로 흡수됨).
m = mujoco.MjModel.from_xml_string(XML)
d = mujoco.MjData(m)
m_warp = mjwarp.put_model(m)
d_warp = mjwarp.put_data(m, d)
d_warp.ctrl.fill_(5.0)

N = 50
I_sim = np.zeros(N)
qvel_sim = np.zeros(N)
F_sim = np.zeros(N)

for k in range(N):
    mjwarp.step(m_warp, d_warp)
    I_sim[k] = d_warp.act.numpy()[0,0]
    qvel_sim[k] = d_warp.qvel.numpy()[0,0]
    F_sim[k] = d_warp.actuator_force.numpy()[0,0]

# Method A 이산식 (back-EMF는 ctrl=5에 이미 반영되어 있다고 가정)
I_methodA = np.zeros(N)
I_prev = 0.0
for k in range(N):
    I_prev = beta_methodA * I_prev + one_minus_beta * 5.0
    I_methodA[k] = I_prev

# Vanilla filterexact 이산식 (β = exp(-h/τ))
beta_vanilla = np.exp(-dt/tau_e)
I_vanilla_analytic = np.zeros(N)
I_prev = 0.0
for k in range(N):
    I_prev = beta_vanilla * I_prev + (1-beta_vanilla) * 5.0
    I_vanilla_analytic[k] = I_prev

t_ms = (np.arange(N) + 1) * dt * 1e3

fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax[0].plot(t_ms, I_sim, 'b-', lw=2, label='mjwarp Method A (sim)')
ax[0].plot(t_ms, I_methodA, 'r--', lw=1.2, label=f'Method A analytic (β={beta_methodA:.3f})')
ax[0].plot(t_ms, I_vanilla_analytic, 'k:', lw=1, label=f'vanilla filterexact (β={beta_vanilla:.3f})')
ax[0].axhline(5.0, color='gray', ls=':', alpha=0.4, label='ctrl = 5A')
ax[0].set_ylabel('Current I [A]')
ax[0].legend()
ax[0].grid(alpha=0.3)
ax[0].set_title('Method A current tracking — sim vs analytic')

residual = I_sim - I_methodA
ax[1].plot(t_ms, residual, 'g-', lw=1)
ax[1].set_ylabel('I_sim − I_analytic [A]')
ax[1].set_xlabel('time [ms]')
ax[1].grid(alpha=0.3)
ax[1].set_title(f'Residual (max abs: {np.max(np.abs(residual)):.2e} A)')
plt.tight_layout()
plt.show()

print(f'\n패치된 mjwarp이 Method A 이산식을 정확히 따르고 있음.')
print(f'Vanilla과는 시정수 미묘하게 다름 (β: {beta_methodA:.3f} vs {beta_vanilla:.3f}).')

## 6. Schur 항 효과 토글 비교

**dynprm[1] = 0**으로 설정하면 패치들이 motor branch를 건너뛰고 vanilla filterexact 동작으로 회귀합니다 (Schur, RHS 보정, Method A 적분기 모두 비활성화).

→ dt=0.1ms에서는 차이가 작지만, 큰 dt에서는 Schur 없는 경우 oscillation/instability가 나타날 수 있음.

In [ ]:
# Source: 자체 작성. 토글 메커니즘 출처:
#   - dynprm[1]=0 (Ke·gr 0) → motor coupling 비활성 분기
#       mjwarp/_src/forward.py:147-160     (_next_act: 조건문 'dynprm[1] != 0.0')
#       mjwarp/_src/forward.py:752-770     (_actuator_force RHS 보정: 같은 조건)
#       mjwarp/_src/derivative.py:67-83    (Schur scale: 'Ke_gr != 0.0' 조건)
#       → 셋 다 false 시 vanilla filterexact 그대로 (β=exp(-h/τ), Schur=0, RHS 보정 없음)
#   - dynprm[2]=0 (L=0)도 같은 motor branch 비활성 — 안전하게 둘 다 0
#   - 의도된 활성화 조건: docs/method_a_mjwarp_patch.md §6
def simulate(dynprm_str, dt_sim=0.0001, N_steps=100, ctrl=5.0):
    xml_v = f'''
    <mujoco>
      <option timestep="{dt_sim}" integrator="implicitfast"/>
      <worldbody>
        <body name="link">
          <joint name="j" type="hinge" axis="0 0 1"/>
          <geom type="sphere" size="0.1" mass="1"/>
        </body>
      </worldbody>
      <actuator>
        <general joint="j" dyntype="filterexact"
                 dynprm="{dynprm_str}"
                 gainprm="{Kt_gr}" biasprm="0 -20 -1"/>
      </actuator>
    </mujoco>
    '''
    m = mujoco.MjModel.from_xml_string(xml_v)
    d = mujoco.MjData(m)
    mw = mjwarp.put_model(m); dw = mjwarp.put_data(m, d)
    dw.ctrl.fill_(ctrl)
    I_arr, qv_arr, F_arr = [], [], []
    for _ in range(N_steps):
        mjwarp.step(mw, dw)
        I_arr.append(dw.act.numpy()[0,0])
        qv_arr.append(dw.qvel.numpy()[0,0])
        F_arr.append(dw.actuator_force.numpy()[0,0])
    return np.array(I_arr), np.array(qv_arr), np.array(F_arr)

# Method A: motor coupling 활성 (dynprm[1] = Ke·gr)
I_A, qv_A, F_A = simulate(f'{tau_e} {Ke_gr} {L} {Ke_gr}', dt_sim=dt, N_steps=100)

# Vanilla: motor coupling 비활성 (dynprm[1]=0, dynprm[2]=0 → 완전한 vanilla)
I_V, qv_V, F_V = simulate(f'{tau_e} 0 0 0', dt_sim=dt, N_steps=100)

t_ms = (np.arange(100) + 1) * dt * 1e3
fig, ax = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
for a, dat_A, dat_V, lbl in zip(ax, [I_A, qv_A, F_A], [I_V, qv_V, F_V], ['Current I [A]', 'qvel [rad/s]', 'Force [N·m]']):
    a.plot(t_ms, dat_A, 'b-', lw=1.5, label='Method A (Schur ON)')
    a.plot(t_ms, dat_V, 'r--', lw=1.2, label='vanilla (Schur OFF)')
    a.set_ylabel(lbl); a.legend(); a.grid(alpha=0.3)
ax[-1].set_xlabel('time [ms]')
ax[0].set_title('Method A vs vanilla filterexact')
plt.tight_layout(); plt.show()

print(f'qvel @ 10ms: Method A = {qv_A[-1]:.3f}, vanilla = {qv_V[-1]:.3f}')
print(f'(vanilla은 Schur 없어 ω가 약간 더 커짐 — back-EMF damping 미반영)')

## 7. Demag 시나리오 — RHS 보정 검증

Demag가 발생하면 plant의 Ke가 줄어들고 (dynprm[1]=Ke_plant·gr) controller는 nominal Ke로 ctrl을 계산합니다 (dynprm[3]=Ke_nom·gr).

→ `act_dot += (dynprm[3] − dynprm[1])·ω/L` 가 활성화되어, controller mismatch만큼 추가 보정.

Demag factor = 0.5 (Ke_plant = 0.5·Ke_nom)으로 시뮬레이션.

In [ ]:
# Source: 자체 작성. Demag 보정 식 출처:
#   - act_dot += (dynprm[3] - dynprm[1])·ω/L
#       mjwarp/_src/forward.py:679-693
#       (이미 4/21에 적용된 기존 패치, Method A 패치와 별도)
#   - 수식 derivation: docs/method_a_mjwarp_patch.md §4
#       (controller가 Ke_nom 가정한 V를 plant Ke_plant에 인가했을 때의 mismatch)
#   - inject_demagnetization 실제 사용 패턴:
#       scripts/eval_demag_v2.py:55-82
#       (RR_calf 등 특정 actuator의 dynprm[1]을 NOMINAL_KE_GR * demag로 덮어씀,
#        gainprm[0]도 Kt_plant·gr로 동시 변경)
# 본 셀에서는 단순화: dynprm[1]만 0.5·Ke_nom으로 빌드 단계에서 직접 설정.
# Healthy: dynprm[1] == dynprm[3]
I_h, qv_h, F_h = simulate(f'{tau_e} {Ke_gr} {L} {Ke_gr}', dt_sim=dt, N_steps=200)

# Demag 50%: Ke_plant = 0.5·Ke_nom
Ke_plant_gr = 0.5 * Ke_gr
I_d, qv_d, F_d = simulate(f'{tau_e} {Ke_plant_gr} {L} {Ke_gr}', dt_sim=dt, N_steps=200)

t_ms = (np.arange(200) + 1) * dt * 1e3
fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax[0].plot(t_ms, I_h, 'b-', lw=1.5, label='healthy')
ax[0].plot(t_ms, I_d, 'r--', lw=1.2, label='demag 50% (Ke_plant=0.5·Ke_nom)')
ax[0].set_ylabel('I [A]'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[0].set_title('Demag effect through dynprm[3] − dynprm[1] correction')
ax[1].plot(t_ms, qv_h, 'b-', lw=1.5, label='healthy')
ax[1].plot(t_ms, qv_d, 'r--', lw=1.2, label='demag 50%')
ax[1].set_ylabel('qvel [rad/s]'); ax[1].set_xlabel('time [ms]')
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Demag 시 ω가 healthy보다 빠르게 증가 (back-EMF damping 약해짐).')
print(f'qvel @ 20ms: healthy = {qv_h[-1]:.3f}, demag = {qv_d[-1]:.3f}')

## 8. 정리

**검증된 것:**
1. ✅ `_next_act`가 Method A β=1/(1+h/τ) 적분기 사용 (Cell 5의 residual ~1e-15)
2. ✅ RHS 보정이 force = Kt·gr·I_predicted 만들어 줌 (Cell 4)
3. ✅ Schur 항이 implicit M_eff에 들어가 ω 발산 억제 (Cell 6 vanilla 비교)
4. ✅ Demag 시 dynprm[3]−dynprm[1] 보정 작동 (Cell 7)

**다음 단계:**
- Go2 환경에서 `Unitree-Go2-Flat-MethodA-Electric` task로 짧은 학습 (~100 iter)
- 기존 vanilla filterexact 학습 baseline과 reward curve 비교
- 문제 없으면 본 학습 (10001 iter)